# Immune integration: CRE selection

Select cell-type-associated cis-regulatory elements (CREs) using annotation-based affinity analysis.
This notebook is a **draft/stub** — the affinity analysis implementation is to be completed by a separate agent.

**Reference**: `cell2state_embryo/notebooks/benchmark/section_suspension_embryo_10x_kit/affinity_analysis_from_annotation_v2_sections.ipynb`

**Prerequisites**:
- NB3 (ATAC data): `adata_atac_tiles.h5ad`
- NB5 (RNA model): trained model with UMAP in `results/immune_integration_rna/model/outputs/`
- NB5.5A (annotations): `annotation_transfer.csv`

**Output**: CRE selection table (to be used by NB6 multimodal training)

In [ ]:
import scanpy as sc
import pandas as pd
from matplotlib import rcParams
import os

rcParams["pdf.fonttype"] = 42

## Parameters

In [ ]:
# Papermill parameters
results_folder = "results/immune_integration/"
rna_model_folder = "results/immune_integration_rna/model/"
atac_path = "results/immune_integration/adata_atac_tiles.h5ad"
k_neighbors = 50

## Annotation hierarchy

The annotation hierarchy has 4 levels, from finest to coarsest:

| Level | Description | N categories | Examples |
|-------|-------------|--------------|----------|
| `harmonized_name` | Original fine annotation | 68 | CD4+ T naive, CD14+ Mono, NK CD56bright |
| `level_1` | Finest grouping | ~20 | CD4+ T-cell lineage, Monocyte lineage, NK-cell lineage |
| `level_2` | Cell type families | ~12 | T-cell lineage, B-cell lineage, Myeloid lineage |
| `level_3` | Broad lineages | ~5 | Lymphoid lineage, Myeloid lineage, Erythroid / MK lineage |
| `level_4` | Root | 1 | Hematopoietic lineage |

### Key cell type groups

| Group | level_2 | level_1 subtypes |
|-------|---------|------------------|
| **T cells** | T-cell lineage | CD4+ T-cell lineage, CD8+ T-cell lineage, MAIT, gamma-delta T, double-negative T |
| **B cells** | B-cell lineage | Naive B, Developing B, Memory B, Activated B, Plasma B cell, B1 B |
| **NK/ILC** | NK-cell lineage | NK-cell lineage (includes NK, NK CD56bright, ILC) |
| **Monocytes** | Myeloid lineage | Monocyte lineage (CD14+, CD16+, Proinflammatory) |
| **DCs** | Myeloid lineage | DC lineage (cDC, cDC2, pDC, ASDC) |
| **HSCs** | HSCs, Lymph prog, MK/E prog | Individual progenitor types |
| **Erythroid** | Erythroid lineage | Proerythroblast, Erythroblast, Normoblast |

CRE selection should identify ATAC peaks/tiles that are differentially accessible between these groups.
The affinity analysis compares each annotation group against appropriate background groups defined per hierarchy level.

## Load RNA data with NB5 UMAP

In [ ]:
# Load RNA adata
rna_path = os.path.join(results_folder, "adata_rna.h5ad")
adata_rna = sc.read_h5ad(rna_path)
print(f"RNA: {adata_rna.shape}")

# Load NB5 UMAP
output_dir = os.path.join(rna_model_folder, "outputs")
X_umap = pd.read_csv(os.path.join(output_dir, f"X_umap_k{k_neighbors}.csv"), index_col=0)
X_scVI = pd.read_csv(os.path.join(output_dir, "X_scVI.csv"), index_col=0)

# Align
common_cells = adata_rna.obs_names.intersection(X_umap.index)
adata_rna = adata_rna[common_cells].copy()
adata_rna.obsm["X_umap"] = X_umap.loc[common_cells].values
adata_rna.obsm["X_scVI"] = X_scVI.loc[common_cells].values
print(f"RNA with UMAP: {adata_rna.shape}")

## Load ATAC data

In [ ]:
# Load ATAC tiles
adata_atac = sc.read_h5ad(atac_path)
print(f"ATAC: {adata_atac.shape}")

# Intersect cells between RNA and ATAC
common_cells = adata_rna.obs_names.intersection(adata_atac.obs_names)
print(f"Common cells (RNA ∩ ATAC): {len(common_cells)}")

adata_rna_sub = adata_rna[common_cells].copy()
adata_atac_sub = adata_atac[common_cells].copy()

# Copy UMAP to ATAC for visualization
adata_atac_sub.obsm["X_umap"] = adata_rna_sub.obsm["X_umap"]
print(f"ATAC subset: {adata_atac_sub.shape}")

## Load annotation transfer

In [ ]:
# Load annotation hierarchy CSV from NB5.5A
transfer_path = os.path.join(results_folder, "annotation_transfer.csv")
transfer_df = pd.read_csv(transfer_path, index_col=0)
print(f"Annotation transfer: {len(transfer_df)} cells")
print(f"Clusters: {transfer_df['cluster_id'].nunique()}")

# Join to ATAC adata
common_annotated = adata_atac_sub.obs_names.intersection(transfer_df.index)
print(f"ATAC cells with annotations: {len(common_annotated)}")

for col in transfer_df.columns:
    adata_atac_sub.obs[col] = transfer_df.loc[common_annotated, col].reindex(adata_atac_sub.obs_names)

print(f"\nLevel 2 categories: {adata_atac_sub.obs['transferred_level_2'].nunique()}")
print(adata_atac_sub.obs["transferred_level_2"].value_counts())

## Load hierarchy lookup

In [ ]:
# Parse annotation_hierarchy.md
hierarchy_path = "docs/notebooks/immune_integration/annotation_hierarchy.md"
hierarchy_rows = []

with open(hierarchy_path) as f:
    for line in f:
        line = line.strip()
        if not line.startswith("|") or line.startswith("| ---") or line.startswith("| harmonized"):
            continue
        parts = [p.strip() for p in line.split("|")[1:-1]]
        if len(parts) < 5:
            continue
        name, l1, l2, l3, l4 = parts[:5]
        if name.startswith("**"):
            continue
        hierarchy_rows.append(
            {
                "harmonized_name": name,
                "level_1": l1,
                "level_2": l2,
                "level_3": l3,
                "level_4": l4,
            }
        )

hierarchy_df = pd.DataFrame(hierarchy_rows)
print(f"Hierarchy: {len(hierarchy_df)} cell types")
display(hierarchy_df)

## UMAP visualization

In [ ]:
palette = sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy

rcParams["figure.figsize"] = 7, 7
sc.pl.umap(
    adata_atac_sub,
    color=["transferred_level_2", "transferred_level_1", "transferred_harmonized_annotation"],
    ncols=1,
    palette=palette,
    size=1,
    legend_fontsize=8,
)

## CRE selection (TO BE IMPLEMENTED)

The following cells are stubs for the affinity analysis workflow.
Reference: `cell2state_embryo/notebooks/benchmark/section_suspension_embryo_10x_kit/affinity_analysis_from_annotation_v2_sections.ipynb`

### Steps:
1. Build `reference_groups` dict defining comparison groups per hierarchy level
2. Build binary annotation masks (`obsm['annotation_selection_with_backgrounds']`)
3. Run affinity analysis (`cell2state.compute_annotation_distance_in_groups`)
4. Threshold and select top-N CREs per annotation group
5. Save CRE selection table

In [ ]:
# TODO: Build reference_groups dict
# This defines which annotation groups to compare against which backgrounds.
# Example structure:
# reference_groups = {
#     "CD4+ T-cell lineage": {
#         "foreground": ["CD4+ T naive", "CD4+ T activated", ...],
#         "background": ["CD8+ T-cell lineage", "NK-cell lineage", ...],
#     },
#     ...
# }
#
# Groups should be defined at level_1 or level_2, comparing each type
# against biologically relevant backgrounds (e.g., T cells vs other lymphoid,
# lymphoid vs myeloid, etc.)

reference_groups = {}  # TO BE IMPLEMENTED
print("reference_groups: TO BE IMPLEMENTED")

In [ ]:
# TODO: Build binary annotation masks
# For each foreground/background group, create binary masks in obsm
#
# annotation_masks = np.zeros((adata_atac_sub.n_obs, n_groups), dtype=bool)
# For each group:
#   foreground cells get 1, background cells get -1 (or similar encoding)
#
# adata_atac_sub.obsm["annotation_selection_with_backgrounds"] = annotation_masks

print("annotation_selection_with_backgrounds: TO BE IMPLEMENTED")

In [ ]:
# TODO: Run affinity analysis
# from cell2state.utils.annotation_analysis import compute_annotation_distance_in_groups
#
# results = compute_annotation_distance_in_groups(
#     adata_atac_sub,
#     annotation_key="annotation_selection_with_backgrounds",
#     ...
# )

print("affinity_analysis: TO BE IMPLEMENTED")

In [ ]:
# TODO: Threshold and select CREs
# Select top-N CREs per annotation group based on affinity scores
# Save as DataFrame: tile_id, annotation_group, affinity_score, rank
#
# cre_selection = pd.DataFrame(...)
# cre_path = os.path.join(results_folder, "cre_selection.csv")
# cre_selection.to_csv(cre_path, index=False)

print("cre_selection: TO BE IMPLEMENTED")